---
title: "Meshing a packed-bed pore space with a volume-controlled Voronoi grid"
subtitle: "Seed the interstitial space of a peclet.dem sphere packing, tessellate + clip against the sphere walls, and relax the seeds so the cell volumes follow a target — uniform, or graded to refine at the walls (an inflation layer)."
author: "Peclet"
date: "2026-07-04"
categories: [voro, dem, meshing, porous-media, sdf]
jupyter: python3
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/computational-chemical-engineering/peclet-examples/blob/main/examples/pore-mesh-voronoi/index.ipynb){target="_blank"}
&nbsp;Runs on a free Colab CPU runtime — the first cell installs `peclet` from PyPI.

::: {.callout-note}
**Experimental API.** The SDF-walled pore-mesh functions used here — `peclet.voro.optimize_pore_mesh`
and `peclet.voro.sdf_voronoi_cells` — are new and still evolving (see the open issues at the end). The
packing (`peclet.dem`) is stock.
:::

## What you'll learn

How to build a **body-fitted Voronoi mesh of a pore space**: drop seed points into the fluid between
packed spheres, tessellate, **clip every cell against the sphere surfaces** (their signed-distance
field), then *move the seeds* so the cell volumes hit a per-cell target `V_ref`. With a wall-graded
target you get a **boundary-layer (inflation-layer) mesh** — small cells hugging the walls, coarser
cells in the open pores — the Voronoi analogue of a graded CFD mesh. You'll also see, honestly, where
the position-only relaxation still struggles (cell collapse in tight throats), which motivates the
next step.

## The setup

The packing is periodic; the fluid is everything outside the spheres. The solid is described by the
union signed-distance field $\phi(\mathbf{x}) = \min_i(\lVert \mathbf{x}-\mathbf{c}_i\rVert - r_i)$
($\phi<0$ inside a sphere, $>0$ in the fluid). We seed $N$ points where $\phi>0$, build their Voronoi
cells with `peclet.voro`, and clip each against the $\phi=0$ surfaces so the cells tile only the fluid.

We then minimise a volume energy over the seed positions. The **relative** form
$E=\sum_i (V_i/V_{\mathrm{ref},i}-1)^2$ drives each cell to its target; the **free-energy** form

$$
E = -\sum_i V_{\mathrm{ref},i}\,\log V_i
$$ {#eq-free}

is especially natural — its constant-"pressure" equilibrium ($\partial E/\partial V_i = -V_{\mathrm{ref},i}/V_i$
equal for all $i$, with $\sum_i V_i$ fixed by the tessellation) is exactly $V_i \propto V_{\mathrm{ref},i}$,
and the $-\log V$ term resists collapse. For the graded target we use
$V_{\mathrm{ref}} = \mathrm{clamp}(\phi, s_{lo}, s_{hi})^3$: a cell *volume* that shrinks toward the wall.

In [ ]:
#| label: bootstrap
#| code-summary: "Environment bootstrap (installs peclet from PyPI on Colab/Binder)"
import importlib.util, os, subprocess, sys
_local = os.environ.get("PECLET_LOCAL_BUILD")
if _local:
    sys.path[:0] = _local.split(os.pathsep)                     # local dem/voro build(s)
elif importlib.util.find_spec("peclet") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peclet"], check=True)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from matplotlib.patches import Circle
from matplotlib.colors import LogNorm, TwoSlopeNorm
from peclet import voro
from peclet_examples import pore_mesh as pm

plt.rcParams.update({"figure.dpi": 130, "font.size": 10, "figure.facecolor": "white",
                     "savefig.bbox": "tight"})

## Step 1 — pack the spheres

`peclet.dem` grows a random close packing (porosity ≈ 0.37).

In [ ]:
#| label: pack
centres, radii, L = pm.pack_spheres(n=180)
print(f"{len(centres)} spheres, box L={L:.2f}, porosity ≈ {1 - (4/3*np.pi*(radii**3).sum())/L**3:.2f}")

## Step 2 — seed, tessellate, relax (four stages)

Each stage seeds the pore space, then (for the relaxed stages) calls `optimize_pore_mesh` to move the
seeds toward a target volume, and finally `sdf_voronoi_cells` to get the clipped cell polyhedra. The
seeding is a **bulk cloud plus an inflation layer** (`seed_pore_space`) — a shell of seeds hugging every
sphere. Without it, a uniform cloud under-samples the thin near-wall fluid and the flat-facet SDF clip
recedes from the curved walls, leaving a bright rim around each sphere in the cross-section; the
inflation layer meshes the walls so the pore space fills.

- **1 random** — uniform bulk + inflation layer, raw Voronoi.
- **2 uniform relaxation** — stage-1 seeds relaxed toward one target volume.
- **3 graded seeding** — bulk density $\propto 1/V_{\mathrm{ref}}$ + inflation layer, raw.
- **4 graded relaxation** — stage-3 seeds relaxed toward $V_{\mathrm{ref}}=\phi^3$.

In [ ]:
#| label: run
opt = dict(method="steepest", max_iter=40, sw=6)                # position-only descent on the walls

s1 = pm.seed_pore_space(centres, radii, L, n_bulk=2000, graded=False, wall_per=70, seed=1)
cells1 = voro.sdf_voronoi_cells(s1, centres, radii, L)

vset_u = np.full(len(s1), L**3 / len(s1))
s2 = voro.optimize_pore_mesh(s1, vset_u, centres, radii, L, **opt)["positions"]
cells2 = voro.sdf_voronoi_cells(np.ascontiguousarray(s2), centres, radii, L)

s3 = pm.seed_pore_space(centres, radii, L, n_bulk=2000, graded=True, wall_per=70, seed=3)
vset3 = pm.vref_graded(pm.union_sdf(s3, centres, radii, L))
cells3 = voro.sdf_voronoi_cells(s3, centres, radii, L)

s4 = voro.optimize_pore_mesh(s3, vset3, centres, radii, L, **opt)["positions"]
s4 = np.ascontiguousarray(s4)
vset4 = pm.vref_graded(pm.union_sdf(s4, centres, radii, L))
cells4 = voro.sdf_voronoi_cells(s4, centres, radii, L)
print("cells per stage:", [len(c["volume"]) for c in (cells1, cells2, cells3, cells4)])

## Step 3 — a cross-section of each stage

We slice the 3-D cells at $z=L/2$ and colour them by cell volume (grey discs = spheres cut by the
plane). The slicing is a small numpy helper on the returned polyhedra — no VTK needed.

In [ ]:
#| label: fig-volume
#| fig-cap: Cross-section of the four stages, cells coloured by volume (log, shared scale). The raw seeded stages (1, 3) fill the pore space up to the walls thanks to the inflation layer; the relaxed stages (2, 4) partly collapse cells in the tight throats (gaps).
z0 = L / 2
allc = [cells1, cells2, cells3, cells4]
titles = ["1 · random seeds", "2 · uniform relaxation", "3 · graded seeding  Vref=φ³", "4 · graded relaxation"]
sl = [pm.tile_periodic(*pm.slice_cells(c, z0), L) for c in allc]   # +periodic images at the box faces
allv = np.concatenate([v for _, v in sl]); norm = LogNorm(*np.percentile(allv[allv > 0], [3, 97]))

def draw(ax, polys, vals, nrm, cmap, title, win=None, lw=0.25):
    ax.add_collection(PolyCollection(polys, array=np.asarray(vals), cmap=cmap, norm=nrm,
                                     edgecolors="k", linewidths=lw, alpha=0.92))
    for c, r in zip(centres, radii):
        if abs(z0 - c[2]) < r:
            for sx in (0, -L, L):
                for sy in (0, -L, L):
                    ax.add_patch(Circle((c[0]+sx, c[1]+sy), np.sqrt(r*r-(z0-c[2])**2),
                                        facecolor="0.55", edgecolor="0.25", lw=0.5, zorder=3))
    ax.set(xlim=win[:2] if win else (0, L), ylim=win[2:] if win else (0, L), aspect="equal")
    ax.set_title(title, fontsize=10); ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(2, 2, figsize=(11, 11))
for ax, (polys, vals), t in zip(axes.ravel(), sl, titles):
    draw(ax, polys, vals, norm, "viridis", t)
fig.colorbar(axes[0, 0].collections[0], ax=axes, fraction=0.025, pad=0.02, label="cell volume (log)")
plt.show()

Structurally, the **raw seeded meshes (1, 3) tile the pore space cleanly** up to the spheres — the
inflation layer is what fills the near-wall band. Graded seeding (3) additionally gives the intended
ring of small cells around every sphere with larger cells in the open pores. The relaxed stages move
the seeds toward equal (2) / graded (4) volumes but also lose cells to collapse in the throats — the
remaining gaps (open issue 1). Any faint scalloping right at a sphere is the flat-facet (tangent-plane)
SDF clip approximating the curved surface; it shrinks with a denser inflation layer.

## Step 4 — what happens at a wall

Zooming onto one sphere makes the wall behaviour explicit — the honest headline of this example.

In [ ]:
#| label: fig-wall
#| fig-cap: 'Zoom at a sphere. Left: random seeds tile the pore space cleanly. Right: after graded relaxation the tight throats develop gaps — cells there are squeezed to zero volume.'
cross = [(i, r*r-(z0-c[2])**2) for i, (c, r) in enumerate(zip(centres, radii)) if abs(z0-c[2]) < r]
i0 = max(cross, key=lambda t: t[1])[0]; cx, cy, w = centres[i0, 0], centres[i0, 1], 1.6
win = (cx-w, cx+w, cy-w, cy+w)
fig, ax = plt.subplots(1, 2, figsize=(11, 5.5))
for a, k, t in [(ax[0], 0, "random seeds"), (ax[1], 3, "graded seeding + relaxation")]:
    draw(a, *sl[k], norm, "viridis", t, win=win, lw=0.5)
plt.show()

## Results — equalisation quality

Colouring the two *relaxed* meshes by $V/V_{\mathrm{ref}}$ (renormalised so 1 = on target) shows how
well each target was met. The **graded target is broadly achieved** — a boundary layer of near-target
cells, with red only in throats it can't fill — while the **uniform target is not**: the pore
geometry is too heterogeneous for equal-volume cells without moving seeds *between* pores.

In [ ]:
#| label: fig-rel
#| fig-cap: V/V_ref after relaxation (blue = too small, white = on target, red = too big). Graded (right) is broadly on target; uniform (left) is not.
def rel(cells, vref):
    v = np.asarray(cells["volume"]); vr = np.asarray(vref)[np.asarray(cells["seed"])]
    return v / (vr * v.sum() / vr.sum())           # renormalise so Σ V_ref = Σ V (on-target = 1)

fig, ax = plt.subplots(1, 2, figsize=(11, 5.5))
rn = TwoSlopeNorm(vmin=1/4, vcenter=1.0, vmax=4.0)
for a, cells, vref, t in [(ax[0], cells2, vset_u, "uniform target"),
                          (ax[1], cells4, vset4, "graded target φ³")]:
    draw(a, *pm.tile_periodic(*pm.slice_cells(cells, z0, rel(cells, vref)), L), rn, "coolwarm", t)
fig.colorbar(ax[1].collections[0], ax=ax, fraction=0.025, pad=0.02, label="V / V_ref")
plt.show()

## Adapt this yourself

- **Change the grading.** Edit `pm.vref_graded` — e.g. $\phi$ (linear) or $\phi^2$ for a thicker
  inflation layer, or a *field* target (finer where a solution gradient is large) for solution-adaptive
  meshing. `optimize_pore_mesh` takes an arbitrary per-cell `vref`.
- **Denser or coarser.** Raise `n_bulk` for the interior and `wall_per` (seeds per sphere) for the
  inflation layer; shrink `wall_eps` to push that layer tighter to the walls.
- **Method.** `optimize_pore_mesh(..., method="graphamg"|"jacobi"|"colored_gs")` for Gauss–Newton, or
  `free_energy=True` for the ideal-gas objective; `mu_barrier>0` adds a log-barrier against collapse.

## Open issues (logged in ISSUES.md)

1. **Cell collapse in tight throats.** Position-only relaxation can't move seeds *between* pores, so
   for an unmatched seeding it collapses cells instead of redistributing. Density-graded seeding
   (stage 3) is the mitigation; a hard log-barrier keeps cells open once the start is feasible.
2. **Curved-wall gradient.** The cell-volume gradient's wall term is exact for a flat wall but only
   first-order for a sphere; with the delicate free-energy gradient this stalls the relaxation. An
   exact tessellator-side wall gradient is the fix.

*(Some gaps in the figures are also a reconstruction artefact — `sdf_voronoi_cells` rebuilds each
clipped cell independently, and a large open-pore cell can fail to re-bound.)*

## Reproduce this

```bash
pip install -e .          # + a recent peclet (peclet-voro with optimize_pore_mesh)
quarto render examples/pore-mesh-voronoi/index.qmd --execute
# local suite build instead of PyPI (dem + voro build dirs, os.pathsep-joined):
PECLET_LOCAL_BUILD=/path/to/suite/dem/build:/path/to/suite/voro/build_py \
  quarto render examples/pore-mesh-voronoi/index.qmd --execute
```